In [123]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from src.agentic.tools import NOTEBOT_TOOLS, _NOTE_STORE
from dotenv import load_dotenv
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.agents.middleware import (wrap_tool_call,before_model,after_model,AgentState)
from collections.abc import Callable
from langchain.tools.tool_node import ToolCallRequest
from langgraph.runtime import Runtime
from typing import Any
from langgraph.stream import StreamTransformer, StreamChannel
import pprint
pprint = pprint.pprint

In [124]:
load_dotenv()

True

In [125]:
model = ChatOllama(
    model="gemma4:e2b-mlx",
    reasoning=False,
    temperature=0.0,
    top_p=0.5
)

In [126]:
model = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url=os.getenv("OLLAMA_BASE_URL"),
    headers={"Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}"},
    temperature=0
)

In [127]:
@wrap_tool_call
def log_tool(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage | Command],) -> ToolMessage | Command:
    print(f"Executing tool: {request.tool_call['name']}")
    print(f"Arguments: {request.tool_call['args']}")
    try:
        result = handler(request)
        print(f"Tool result:{result.content[:40]}...")
        return result
    except Exception as e:
        print(f"Tool failed: {e}")
        raise

In [128]:
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"About to call model with {len(state['messages'])} messages")
    return None

In [129]:
@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]

    if not isinstance(last, AIMessage):
        return None

    for tool_call in last.tool_calls:
        if tool_call["name"] == "search_notes":
            print("Search tool intercepted.")
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content="Search is currently unavailable. Try again after 5mins",
                            tool_call_id=tool_call["id"],
                        )
                    ]
                }
            )

    # No interception → continue normally
    return None

In [130]:
agent = create_agent(
    model=model,
    system_prompt="You are a NoteBot. each tool you have is designed for processing a single note you need to make multiple tool calls if you want to process more than one note.",
    tools=NOTEBOT_TOOLS,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'save_note':True,
                'delete_note':{
                    "allowed_decisions":['approve','reject']
                }
            },
            description_prefix="Tool execution pending approval"
        ),
        ToolRetryMiddleware(
            max_retries=9,
            backoff_factor=1.1
        ),
        log_tool,
        log_before_model,
    ],
)

In [131]:
config = {"configurable": {"thread_id": "test0"}}

In [132]:
class MyCustomTransformer(StreamTransformer):
    required_stream_modes = ("custom",)

    def __init__(self, scope: tuple[str, ...] = ()) -> None:
        super().__init__(scope)
        # 1. Define the channel
        self.log = StreamChannel()

    def init(self) -> dict:
        # 2. Key "my_custom" will be the name used in interleave()
        return {"my_custom": self.log}

    def process(self, event) -> bool:
        if event["method"] == "custom":
            self.log.push(event["params"]["data"])
        return True

In [99]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for message in res.messages:
        for chunk in message.text:
            print(chunk,end="",flush=True)
        print("\n")
    msg=input("\nEnter: ")

In [133]:
msg = input("Enter: ")

while msg:
    stream = agent.stream_events(
        {"messages": [HumanMessage(msg)]},
        config=config,
        transformers=[MyCustomTransformer],
        version="v3",
    )
    for name, item in stream.interleave("my_custom", "messages"):
        if name == "my_custom":
            print(f"Tool update: {item}")
        elif name == "messages":
            for j in item.text:
                print(j,end="",flush=True)
    print()
    while stream.interrupted:
        interrupt = stream.interrupts[-1].value

        request = interrupt["action_requests"][-1]
        review = interrupt["review_configs"][-1]

        print("\nAction:")
        print(request["description"])

        print("\nAllowed decisions:")
        for d in review["allowed_decisions"]:
            print("-", d)

        choice = input("\nYour choice: ").strip()

        if choice == "approve":
            decision = {
                "type": "approve"
            }

        elif choice == "reject":
            feedback = input("Reason (optional): ")
            if feedback:
                feedback = f"Unsuccessful. The user has rejected the tool call with the following Feedback:{feedback}. Try Again"

            decision = {
                "type": "reject",
                "message": feedback
            }

        elif choice == "respond":
            reply = input("Response to tool: ")
            decision = {
                "type": "respond",
                "message": reply
            }

        elif choice == "edit":
            print(f"\nTool: {request['name']}")

            new_args = request["arguments"].copy()

            print("\nEdit arguments (press Enter to keep the current value):\n")

            for key, value in new_args.items():
                new_value = input(f"{key} [{value}]: ").strip()

                if new_value:
                    # Convert to original type where possible
                    try:
                        if isinstance(value, bool):
                            new_args[key] = new_value.lower() in (
                                "true",
                                "1",
                                "yes",
                                "y",
                            )
                        elif isinstance(value, int):
                            new_args[key] = int(new_value)
                        elif isinstance(value, float):
                            new_args[key] = float(new_value)
                        else:
                            new_args[key] = new_value
                    except ValueError:
                        print(f"Invalid value for {key}. Keeping original.")

                    decision = {
                        "type": "edit",
                        "edited_action": {
                            "name": request["name"],
                            "args": new_args,
                        },
                    }

        else:
            print("Invalid choice.")
            continue

        stream = agent.stream_events(
            Command(
                resume={
                    "decisions": [decision]
                }
            ),
            transformers=[MyCustomTransformer],
            config=config,
            version="v3",
        )

        for name, item in stream.interleave("my_custom", "messages"):
            if name == "my_custom":
                print(f"Tool update: {item}")
            elif name == "messages":
                for j in item.text:
                    print(j,end="",flush=True)

    msg = input("\nEnter: ")

About to call model with 1 messages


Action:
Tool execution pending approval

Tool: save_note
Args: {'title': 'apple', 'content': 'iphone'}

Allowed decisions:
- approve
- edit
- reject
- respond
Executing tool: save_note
Arguments: {'title': 'apple', 'content': 'iphone'}
Tool result:Success: Note 'apple' saved....
About to call model with 3 messages
Note 'apple' saved successfully.

In [ ]:
msg = input("Enter: ")

while msg:
    stream = agent.stream_events(
        {"messages": [HumanMessage(msg)]},
        config=config,
        version="v3",
    )

    for message in stream.messages:
        for token in message.text:
            print(token, end="", flush=True)
    print()
    
    while stream.interrupted:
        interrupt = stream.interrupts[-1].value

        request = interrupt["action_requests"][-1]
        review = interrupt["review_configs"][-1]

        print("\nAction:")
        print(request["description"])

        print("\nAllowed decisions:")
        for d in review["allowed_decisions"]:
            print("-", d)

        choice = input("\nYour choice: ").strip()

        if choice == "approve":
            decision = {
                "type": "approve"
            }

        elif choice == "reject":
            feedback = input("Reason (optional): ")
            if feedback:
                feedback = f"Unsuccessful. The user has rejected the tool call with the following Feedback:{feedback}. Try Again"

            decision = {
                "type": "reject",
                "message": feedback
            }

        elif choice == "respond":
            reply = input("Response to tool: ")
            decision = {
                "type": "respond",
                "message": reply
            }

        elif choice == "edit":
            print(f"\nTool: {request['name']}")

            new_args = request["arguments"].copy()

            print("\nEdit arguments (press Enter to keep the current value):\n")

            for key, value in new_args.items():
                new_value = input(f"{key} [{value}]: ").strip()

                if new_value:
                    # Convert to original type where possible
                    try:
                        if isinstance(value, bool):
                            new_args[key] = new_value.lower() in (
                                "true",
                                "1",
                                "yes",
                                "y",
                            )
                        elif isinstance(value, int):
                            new_args[key] = int(new_value)
                        elif isinstance(value, float):
                            new_args[key] = float(new_value)
                        else:
                            new_args[key] = new_value
                    except ValueError:
                        print(f"Invalid value for {key}. Keeping original.")

                    decision = {
                        "type": "edit",
                        "edited_action": {
                            "name": request["name"],
                            "args": new_args,
                        },
                    }

        else:
            print("Invalid choice.")
            continue

        stream = agent.stream_events(
            Command(
                resume={
                    "decisions": [decision]
                }
            ),
            config=config,
            version="v3",
        )

        for message in stream.messages:
            for token in message.text:
                print(token, end="", flush=True)
        print()

    msg = input("\nEnter: ")

In [17]:
pprint(_NOTE_STORE)

{}


In [42]:
pprint(agent.get_state(config=config).values)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='77051a14-4a30-4ae8-bc4e-428880b9bc31'),
              AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you with your notes today? I can save, delete, search, or summarize notes for you.', 'index': 0}], additional_kwargs={}, response_metadata={'model': 'gemma4:e2b-mlx', 'created_at': '2026-07-19T06:10:50.262896Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8568990875, 'load_duration': 5500079500, 'prompt_eval_count': 345, 'prompt_eval_duration': 1947076708, 'eval_count': 26, 'eval_duration': 1091903625, 'logprobs': None, 'model_name': 'gemma4:e2b-mlx', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f78ff-8d9c-7051-8354-b11ca0a85ffb', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 345, 'output_tokens': 26, 'total_tokens': 371}),
              HumanMessage(content='find a noted named apple', additional_kwargs={}, response_metada

In [30]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for message in res.messages:
        for chunk in message.text:
            print(chunk,end="",flush=True)
        print("\n")
        for chunk in message.tool_calls:
            print(chunk,end="",flush=True)
    if res.interrupted:
        print(res.interrupts[-1].value['action_requests'][-1]['description'])
        print("\n")
    msg=input("\nEnter: ")




{'type': 'tool_call_chunk', 'id': '1b94b66c-a709-46c8-b00d-d3f2a20f3eb7', 'name': 'search_notes', 'args': '{"query": "orange"}'}The note titled "orange" contains the following content:

"orange is both color and fruit"



In [11]:
pprint(agent.get_state(config=config).values)

{}


In [19]:
pprint(res.interrupts)

[Interrupt(value={'action_requests': [{'args': {'title': 'orange'},
                                       'description': 'Tool execution requires '
                                                      'approval\n'
                                                      '\n'
                                                      'Tool: delete_note\n'
                                                      "Args: {'title': "
                                                      "'orange'}",
                                       'name': 'delete_note'}],
                  'review_configs': [{'action_name': 'delete_note',
                                      'allowed_decisions': ['approve',
                                                            'edit',
                                                            'reject',
                                                            'respond']}]},
           id='51c8eba45f896313440fdac4e479c825')]


In [ ]:
print(res.interrupts[-1].value['action_requests'][-1]['description'])

Tool execution requires approval

Tool: delete_note
Args: {'title': 'orange'}


In [56]:
response = agent.stream_events(
    Command(
        resume={"decisions":[{"type":"approve"}]}
    ),
    config=config,
    version="v3"
)

for i in response.messages:
    for j in i.text:
        print(j,end="",flush=True)

The note titled "orange" has been successfully deleted. Is there anything else you'd like me to help you with?

In [59]:
pprint(agent.get_state(config=config).values)

{'messages': [HumanMessage(content='can you search for notes that have pisces in them?', additional_kwargs={}, response_metadata={}, id='d798e3c6-5d7b-4182-b86c-cb399a02c96d'),
              AIMessage(content=[{'type': 'tool_call', 'id': '1ef6865e-2656-4e66-a3fb-64cf6c679eaa', 'name': 'search_notes', 'args': {'query': 'pisces'}}], additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b-mlx', 'created_at': '2026-07-11T06:36:55.769697Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5014120291, 'load_duration': 1947274041, 'prompt_eval_count': 579, 'prompt_eval_duration': 2095518458, 'eval_count': 25, 'eval_duration': 924863125, 'logprobs': None, 'model_name': 'qwen3.5:4b-mlx', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f4fe4-9ec1-7003-a2f0-24404341ccb3', tool_calls=[{'name': 'search_notes', 'args': {'query': 'pisces'}, 'id': '1ef6865e-2656-4e66-a3fb-64cf6c679eaa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 579, '

In [1]:
pprint(response.interrupted)

Pretty printing has been turned OFF


In [113]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for event in res:
        print("\n")
        print(event)
        print("\n")
    msg=input("\nEnter: ")



{'type': 'event', 'method': 'values', 'params': {'namespace': [], 'timestamp': 1783942697028, 'data': {'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1d6fd6b8-ba8c-4c1e-baf5-d26a334ff77d')]}, 'interrupts': ()}, 'seq': 1}


About to call model with 2 messages


{'type': 'event', 'method': 'messages', 'params': {'namespace': [], 'timestamp': 1783942704776, 'data': ({'event': 'message-start', 'role': 'ai', 'id': 'lc_run--019f5b45-5047-7403-ad8f-6b83b7118674'}, {'ls_integration': 'langchain_chat_model', 'thread_id': 'test0', 'langgraph_step': 2, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:87d51003-3e87-d972-bd23-ebf90ec73ee4', 'checkpoint_ns': 'model:87d51003-3e87-d972-bd23-ebf90ec73ee4', 'ls_provider': 'ollama', 'ls_model_name': 'qwen3.5:4b-mlx', 'ls_model_type': 'chat', 'ls_temperature': 0.0, 'lc_versions': {'langchain-core': '1.4.8', 'lang